# Notebook 5: Analysis (tables + plots)

**Runtime: CPU — no GPU.** Self-contained. Re-plotting never re-runs a model.

## Purpose (one job)
Read `results/*.json` (written by notebook 4) and produce the paper artifacts: the
comparison table and the figures. Because eval and plotting are split, you can
iterate on presentation endlessly without touching a GPU.

In [ ]:
!pip -q install -U pandas matplotlib seaborn

In [ ]:
import os, json, glob
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import os, tempfile
def _load_dotenv(path='.env'):
    # Minimal .env loader (no dependency); real environment vars take precedence.
    try:
        with open(path) as _f:
            for _line in _f:
                _line = _line.strip()
                if not _line or _line.startswith('#') or '=' not in _line:
                    continue
                _k, _v = _line.split('=', 1)
                os.environ.setdefault(_k.strip(), _v.strip().strip('\'"'))
    except FileNotFoundError:
        pass
def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
_load_dotenv()   # pull FEDDAPT_ROOT from .env if present
if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', '/content/drive/MyDrive/FedDAPT')
else:
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', os.path.abspath('./FedDAPT'))
RESULTS_DIR = f'{PROJECT_ROOT}/results'
FIG_DIR = f'{PROJECT_ROOT}/figures'; os.makedirs(FIG_DIR, exist_ok=True)

rows = [json.load(open(p)) for p in glob.glob(f'{RESULTS_DIR}/*.json')]
df = pd.DataFrame(rows)
print(f'{len(df)} results loaded'); df

## 1. Comparison table
Collapse per-client instruction rows to a mean so each method is one line.

In [ ]:
def path_label(r):
    if r['stage'] == 'baseline': return 'zero-shot'
    if r['stage'] == 'dapt':     return f"DAPT-only [{r['method']}, eps={r['epsilon']}]"
    base = r['init_from'] or 'no-DAPT'
    return f"{base} -> instruction"

df['path'] = df.apply(path_label, axis=1)
metrics = ['triage_macro_f1','triage_acc','attack_mc_acc','rouge_l']
table = (df.groupby('path')[metrics].mean().round(3).sort_values('triage_macro_f1'))
table.to_csv(f'{FIG_DIR}/comparison_table.csv')
print(table.to_string()); table

## 2. With vs without instruction tuning (the opt-in comparison)

In [ ]:
sub = df[df['stage'].isin(['dapt','instruction'])].copy()
sub['kind'] = np.where(sub['stage']=='instruction', 'DAPT + instruction', 'DAPT only')
g = sub.groupby('kind')[metrics].mean()
ax = g[['triage_macro_f1','attack_mc_acc','rouge_l']].plot(kind='bar', figsize=(9,5))
ax.set_title('With vs without Stage-2 instruction tuning'); ax.set_ylabel('score')
plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(f'{FIG_DIR}/with_without_instruction.png', dpi=150); plt.show()

## 3. Ablation: does sharing the DAPT help? (A/B/C/D on instruction path)

In [ ]:
instr = df[df['stage']=='instruction'].copy()
def ablation(r):
    s = r['init_from']
    if not s: return 'A: no DAPT'
    if 'fed' in s or 'trimmed' in s or 'krum' in s: return 'C: federated DAPT'
    if 'local' in s: return 'B: local DAPT'
    return s
instr['ablation'] = instr.apply(ablation, axis=1)
order = ['A: no DAPT','B: local DAPT','C: federated DAPT']
gg = instr.groupby('ablation')[['triage_macro_f1']].mean().reindex([o for o in order if o in set(instr['ablation'])])
ax = gg.plot(kind='bar', legend=False, figsize=(7,5), color='#534AB7')
ax.set_title('Federation benefit: instruction-tuned triage macro-F1 by DAPT source')
ax.set_ylabel('triage macro-F1'); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(f'{FIG_DIR}/ablation_dapt_source.png', dpi=150); plt.show()

## 4. Privacy-utility curve (DAPT-only rows, varying epsilon)

In [ ]:
dp = df[(df['stage']=='dapt') & (df['method']=='fedavg')].copy()
dp['eps'] = dp['epsilon'].replace('inf', np.inf).astype(float)
dp = dp.sort_values('eps')
finite = dp[np.isfinite(dp['eps'])]
if len(finite) >= 2:
    plt.figure(figsize=(8,5))
    plt.plot(finite['eps'], finite['triage_macro_f1'], 'o-', color='#534AB7', label='FedDAPT + DP')
    inf = dp[~np.isfinite(dp['eps'])]
    if len(inf): plt.axhline(inf['triage_macro_f1'].iloc[0], ls='--', color='#1D9E75', label='no DP (eps=inf)')
    plt.xlabel('privacy budget epsilon (lower = more private)'); plt.ylabel('triage macro-F1')
    plt.title('Privacy-utility tradeoff'); plt.legend(); plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/privacy_utility.png', dpi=150); plt.show()
else:
    print('Need >=2 finite-eps DAPT runs (run the eps sweep in notebook 2).')

Figures + `comparison_table.csv` are in `FedDAPT/figures/`.